# Notebook 07 — Full-Vocabulary FAISS Vector Retrieval

**Goal:** Remove Notebook 06's 3,000-seed restriction, reuse the **saved** cutoff-specific Item2Vec models, and evaluate full-vocabulary vector retrieval against Notebook 04–06 on the *same warm-user future-view protocol*.

- Validation: only history **before 2015-08-01** and `notebook06_val_item2vec.model`.
- Test: only history **before 2015-09-01** and `notebook06_test_item2vec.model`.
- Never use future labels to build the index or query embeddings.
- This notebook uses normalized mean embeddings from the last 10 distinct viewed items, FAISS inner-product search, excludes **all** previously interacted items, and fills with history-only popularity.
- FAISS `IndexFlatIP` is **exact** vector search; optional `IndexHNSWFlat` provides **approximate nearest-neighbor (ANN)** retrieval. Both are supported below.
- Batch timing is **not** production p95 latency. We'll benchmark serving in Notebook 15.
- RetailRocket has interactions, not impression logs; **not CTR**.

## 1. Dependencies, project paths, and temporal input checks

In [3]:
# Install if necessary in your Jupyter environment, then restart kernel:
# %pip install -U faiss-cpu gensim numpy pandas scipy
from pathlib import Path
import json, time
import numpy as np
import pandas as pd

try:
    import faiss
    from gensim.models import Word2Vec
except ImportError as exc:
    raise ImportError('Install: pip install faiss-cpu gensim numpy pandas scipy; restart kernel') from exc

cwd = Path.cwd().resolve()
if (cwd / 'data' / 'processed').is_dir(): ROOT = cwd
elif (cwd.parent / 'data' / 'processed').is_dir(): ROOT = cwd.parent
else: raise FileNotFoundError(f'Cannot locate data/processed from {cwd}')
PROCESSED, REPORTS, MODELS = ROOT/'data'/'processed', ROOT/'reports', ROOT/'models'
REPORTS.mkdir(exist_ok=True, parents=True)
CUTOFFS = {'val': pd.Timestamp('2015-08-01', tz='UTC'), 'test': pd.Timestamp('2015-09-01', tz='UTC')}
K_VALUES = [10,20,50,100]
MAX_K = 100
inputs = {}
for split, stem in [('val','validation'),('test','test')]:
    h = pd.read_csv(PROCESSED / ('train_history.csv' if split == 'val' else 'test_history.csv'))
    labels = pd.read_csv(PROCESSED / f'{stem}_view_labels_warm_users.csv')
    assert h.timestamp.max() < int(CUTOFFS[split].timestamp()*1000)
    assert labels.visitorid.isin(h.visitorid).all()
    assert not labels[['visitorid','itemid']].duplicated().any()
    assert set(labels.event.unique()) <= {'view'} if 'event' in labels.columns else True
    inputs[split] = {'history':h,'labels':labels}
    print(split, 'history:',len(h),'positive pairs:',len(labels),'users:',labels.visitorid.nunique())
print('PASS: temporal cutoffs and label integrity')

val history: 1899029 positive pairs: 28634 users: 13329
test history: 2452391 positive pairs: 31876 users: 15137
PASS: temporal cutoffs and label integrity


## 2. Load saved Word2Vec embeddings and build full-vocabulary FAISS indices

**Default** `IndexFlatIP` is exact and robust; set `INDEX_KIND="hnsw"` to run approximate HNSW. On normalized vectors, inner product equals cosine similarity. HNSW speed/recall depends on parameters and hardware. Index includes every **Word2Vec vocabulary item** (not every catalog item; `min_count=5` in Notebook 06).

In [4]:
INDEX_KIND = 'flat'  # 'flat' = exact; 'hnsw' = approximate ANN
HNSW_M = 32
HNSW_EF_CONSTRUCTION = 120
HNSW_EF_SEARCH = 128
SEED_ITEMS_PER_USER = 10
SEARCH_K = 300  # over-fetch to allow filtering of seen items
QUERY_BATCH_SIZE = 256

models, indices, vocab_ids, vocab_lookup, build_info = {}, {}, {}, {}, {}
for split in ['val','test']:
    p = MODELS / f'notebook06_{split}_item2vec.model'
    assert p.is_file(), f'Missing saved model: {p}; run Notebook 06 first.'
    t0 = time.perf_counter()
    model = Word2Vec.load(str(p))
    ids = np.asarray([int(t) for t in model.wv.index_to_key], dtype=np.int64)
    vectors = np.ascontiguousarray(model.wv.vectors.astype('float32'))
    faiss.normalize_L2(vectors)
    dimension = vectors.shape[1]
    if INDEX_KIND == 'flat':
        index = faiss.IndexFlatIP(dimension)
    elif INDEX_KIND == 'hnsw':
        index = faiss.IndexHNSWFlat(dimension, HNSW_M, faiss.METRIC_INNER_PRODUCT)
        index.hnsw.efConstruction = HNSW_EF_CONSTRUCTION
        index.hnsw.efSearch = HNSW_EF_SEARCH
    else: raise ValueError('INDEX_KIND must be flat or hnsw')
    index.add(vectors)
    assert index.ntotal == len(ids)
    models[split], indices[split], vocab_ids[split] = model, index, ids
    vocab_lookup[split] = {int(item):i for i,item in enumerate(ids)}
    build_info[split] = {'index_kind':INDEX_KIND, 'vocabulary_items':len(ids),
                         'dimension':dimension, 'index_build_seconds':round(time.perf_counter()-t0,3)}
    print(split, build_info[split])
print('PASS: all learned vocabulary items indexed')

val {'index_kind': 'flat', 'vocabulary_items': 43267, 'dimension': 64, 'index_build_seconds': 0.297}
test {'index_kind': 'flat', 'vocabulary_items': 52280, 'dimension': 64, 'index_build_seconds': 0.329}
PASS: all learned vocabulary items indexed


## 3. Prepare cutoff-safe user histories, popularity fallback, and query vectors

We use each user’s **last 10 distinct viewed items** that exist in the embedding vocabulary. Mean normalized vectors give a simple user-interest representation; it differs from Notebook 06’s *sum of neighbor similarities*, so gains cannot be attributed exclusively to FAISS. For a controlled comparison, a future ablation can use identical scoring with different indices.

In [5]:
def prepare_users(history, labels, lookup, model):
    users = sorted(int(u) for u in labels.visitorid.unique())
    sub = history.loc[history.visitorid.isin(users), ['visitorid','itemid','event','timestamp']]
    seen = sub.groupby('visitorid')['itemid'].agg(lambda x: set(map(int,x))).to_dict()
    views = sub.loc[sub.event.eq('view')].sort_values('timestamp', ascending=False, kind='mergesort')
    views = views.drop_duplicates(['visitorid','itemid'], keep='first')
    seeds = views.groupby('visitorid')['itemid'].agg(lambda x: list(map(int,x.head(SEED_ITEMS_PER_USER)))).to_dict()
    counts = history.loc[history.event.eq('view'),'itemid'].value_counts()
    popularity = (counts.rename_axis('itemid').reset_index(name='n')
                  .sort_values(['n','itemid'],ascending=[False,True],kind='mergesort')
                  .itemid.astype(int).tolist())
    vec = np.asarray(model.wv.vectors, dtype=np.float32)
    queries = np.zeros((len(users),vec.shape[1]),dtype=np.float32)
    has_query = np.zeros(len(users),dtype=bool)
    for j,u in enumerate(users):
        idx = [lookup[i] for i in seeds.get(u,[]) if i in lookup]
        if idx:
            q = vec[idx].mean(axis=0)
            length = np.linalg.norm(q)
            if length > 1e-12:
                queries[j] = q/length
                has_query[j] = True
    return users,seen,popularity,np.ascontiguousarray(queries),has_query

prepared = {}
for split in ['val','test']:
    p = prepare_users(inputs[split]['history'],inputs[split]['labels'],vocab_lookup[split],models[split])
    prepared[split] = p
    print(split,'users with embedding queries:',int(p[4].sum()),'/',len(p[0]))

val users with embedding queries: 10684 / 13329
test users with embedding queries: 12411 / 15137


## 4. Batched FAISS search + seen filtering + popularity fallback

We retrieve `SEARCH_K` raw neighbors, keep positive-similarity unseen items, and fill the rest with historical popularity. Every output is 100 unique items. `SEARCH_K=300` bounds work; heavy-history users may need a larger over-fetch. Track how many users get vector candidates.

In [6]:
results = {}
for split in ['val','test']:
    users, seen, popularity, queries, has_query = prepared[split]
    index, ids = indices[split], vocab_ids[split]
    t0 = time.perf_counter()
    recs, personal_counts = {}, []
    eligible_rows = np.flatnonzero(has_query)
    retrieved = {}
    k_search = min(SEARCH_K, index.ntotal)
    for start in range(0,len(eligible_rows),QUERY_BATCH_SIZE):
        row_ids = eligible_rows[start:start+QUERY_BATCH_SIZE]
        similarities, neighbor_ids = index.search(queries[row_ids], k_search)
        for row, sims, positions in zip(row_ids, similarities, neighbor_ids):
            u = users[int(row)]
            choices = []
            for sim,position in zip(sims,positions):
                if position < 0 or sim <= 0: continue
                item = int(ids[position])
                if item not in seen.get(u,set()):
                    choices.append(item)
                    if len(choices) == MAX_K: break
            retrieved[u] = choices
    for u in users:
        chosen = retrieved.get(u,[])
        personal_counts.append(len(chosen))
        used = set(chosen)
        if len(chosen) < MAX_K:
            for item in popularity:
                if item not in seen.get(u,set()) and item not in used:
                    chosen.append(item); used.add(item)
                    if len(chosen) == MAX_K: break
        assert len(chosen) == MAX_K and len(set(chosen)) == MAX_K
        assert not (set(chosen) & seen.get(u,set()))
        recs[u] = chosen
    elapsed = time.perf_counter()-t0
    results[split] = {'recs':recs,'seen':seen,'users':users,
                      'batch_seconds':elapsed,
                      'users_with_embedding_queries':int(has_query.sum()),
                      'users_with_embedding_candidates':int(sum(c>0 for c in personal_counts)),
                      'avg_embedding_candidates':float(np.mean(personal_counts))}
    print(split,'users:',len(users),'batch_seconds:',round(elapsed,3),
          'vector candidates for:',results[split]['users_with_embedding_candidates'],
          'avg vector candidates:',round(results[split]['avg_embedding_candidates'],2))
# Batch time includes Python filtering/fallback, not production latency.

val users: 13329 batch_seconds: 2.901 vector candidates for: 10684 avg vector candidates: 80.16
test users: 15137 batch_seconds: 3.56 vector candidates for: 12411 avg vector candidates: 81.99


## 5. Evaluate using exactly the Notebook 04–06 metric definitions

All warm-user future-view pairs remain in the recall denominator, including previously seen and cold-item positives. `EligiblePositivePairs` is reported separately, not silently removed.

In [7]:
def evaluate(recs,labels,seen_map,history_items,k_values):
    positives = labels.groupby('visitorid')['itemid'].agg(set).to_dict()
    rows=[]
    for k in k_values:
        recalls,hits,found_total,pos_total=[],[],0,0
        recommended_items=set()
        for u,truth in positives.items():
            selected=set(recs[int(u)][:k])
            found=len(selected & truth)
            recalls.append(found/len(truth)); hits.append(float(found>0))
            found_total+=found; pos_total+=len(truth)
            recommended_items.update(selected)
        rows.append({'K':k,'Recall@K_macro':float(np.mean(recalls)),
                     'Recall@K_micro':found_total/pos_total,
                     'HitRate@K':float(np.mean(hits)),
                     'UniqueRecommendedItems':len(recommended_items),
                     'CatalogCoverage':len(recommended_items)/len(history_items)})
    eligible=sum(1 for u,i in labels[['visitorid','itemid']].itertuples(index=False,name=None)
                 if i in history_items and i not in seen_map.get(int(u),set()))
    return pd.DataFrame(rows),eligible

metrics={}
for split in ['val','test']:
    h=inputs[split]['history']; labels=inputs[split]['labels']
    m,eligible=evaluate(results[split]['recs'],labels,results[split]['seen'],set(h.itemid.unique()),K_VALUES)
    metrics[split]=m; results[split]['eligible_pairs']=eligible
    print('\n',split,'eligible positive pairs:',eligible,'/',len(labels))
    display(m.round(6))


 val eligible positive pairs: 21984 / 28634


,K,Recall@K_macro,Recall@K_micro,HitRate@K,UniqueRecommendedItems,CatalogCoverage
0,10,0.015390,0.013865,0.025433,28787,0.144159
1,20,0.023660,0.022386,0.038488,35099,0.175768
2,50,0.037476,0.037578,0.059344,40313,0.201879
3,100,0.051345,0.051407,0.076675,41919,0.209921



 test eligible positive pairs: 24875 / 31876


,K,Recall@K_macro,Recall@K_micro,HitRate@K,UniqueRecommendedItems,CatalogCoverage
0,10,0.015104,0.013333,0.024443,33860,0.151851
1,20,0.023049,0.020235,0.035806,41538,0.186284
2,50,0.034826,0.032124,0.052190,48186,0.216098
3,100,0.047478,0.044704,0.069300,50436,0.226189


## 6. Compare with earlier retrieval methods

The CSV baselines are optional; the JSON files are a fallback when metrics CSVs are missing. Compare on **validation first**. Test metrics are for reporting, not hyperparameter tuning.

In [8]:
baseline_configs = [('04','Popularity','popularity'),('05','ItemCF','itemcf'),('06','Item2Vec','item2vec')]
comparisons={}
for split in ['val','test']:
    frame=metrics[split][['K','Recall@K_micro','HitRate@K','CatalogCoverage']].rename(
        columns={c:c+'_FAISS' for c in ['Recall@K_micro','HitRate@K','CatalogCoverage']})
    for number,label,stem in baseline_configs:
        csv_path=REPORTS/f'notebook{number}_{split}_{stem}_metrics.csv'
        json_path=REPORTS/f'notebook{number}_{"validation" if split=="val" else "test"}_{stem}.json'
        if number=='05': json_path=REPORTS/'notebook05_itemcf_retrieval.json'
        if number=='06': json_path=REPORTS/'notebook06_item2vec_retrieval.json'
        if number=='04': json_path=REPORTS/'notebook04_popularity_baseline.json'
        if csv_path.is_file():
            other=pd.read_csv(csv_path)
        elif json_path.is_file():
            obj=json.loads(json_path.read_text(encoding='utf-8'))
            other=pd.DataFrame(obj['validation' if split=='val' else 'test']['metrics'])
        else:
            print('Baseline unavailable:',label); continue
        other=other[['K','Recall@K_micro','HitRate@K','CatalogCoverage']].rename(
            columns={c:c+'_'+label for c in ['Recall@K_micro','HitRate@K','CatalogCoverage']})
        frame=frame.merge(other,on='K',how='left',validate='one_to_one')
    comparisons[split]=frame
    print('\n'+split.upper()+' comparison:')
    display(frame.round(6))
    frame.to_csv(REPORTS/f'notebook07_{split}_comparison.csv',index=False)


VAL comparison:


,K,Recall@K_micro_FAISS,HitRate@K_FAISS,CatalogCoverage_FAISS,Recall@K_micro_Popularity,HitRate@K_Popularity,CatalogCoverage_Popularity,Recall@K_micro_ItemCF,HitRate@K_ItemCF,CatalogCoverage_ItemCF,Recall@K_micro_Item2Vec,HitRate@K_Item2Vec,CatalogCoverage_Item2Vec
0,10,0.013865,0.025433,0.144159,0.002968,0.006077,0.000085,0.021443,0.038187,0.244530,0.010372,0.018681,0.058796
1,20,0.022386,0.038488,0.175768,0.005727,0.010728,0.000190,0.032269,0.054018,0.324104,0.015751,0.027684,0.084792
2,50,0.037578,0.059344,0.201879,0.011106,0.019731,0.000431,0.049137,0.078025,0.387493,0.026716,0.042614,0.126782
3,100,0.051407,0.076675,0.209921,0.017636,0.030235,0.000821,0.062373,0.096031,0.405906,0.038102,0.058819,0.150103



TEST comparison:


,K,Recall@K_micro_FAISS,HitRate@K_FAISS,CatalogCoverage_FAISS,Recall@K_micro_Popularity,HitRate@K_Popularity,CatalogCoverage_Popularity,Recall@K_micro_ItemCF,HitRate@K_ItemCF,CatalogCoverage_ItemCF,Recall@K_micro_Item2Vec,HitRate@K_Item2Vec,CatalogCoverage_Item2Vec
0,10,0.013333,0.024443,0.151851,0.002384,0.004955,0.000081,0.019921,0.036863,0.252626,0.009004,0.017110,0.057121
1,20,0.020235,0.035806,0.186284,0.005176,0.010372,0.000188,0.028140,0.049944,0.337870,0.014086,0.025633,0.084065
2,50,0.032124,0.052190,0.216098,0.011325,0.021008,0.000435,0.043544,0.072405,0.407378,0.023246,0.039043,0.129593
3,100,0.044704,0.069300,0.226189,0.017411,0.031314,0.000825,0.054618,0.088591,0.425999,0.034164,0.055559,0.155474


## 7. Save model-independent candidates and metrics for Notebook 08

Save **all** evaluated users’ Top-100 candidates (rather than only 20 sample users) so Hybrid Retrieval can union and score candidates without recomputing FAISS. This is an offline experiment artifact; later we will build a production serving path.

In [9]:
for split in ['val','test']:
    metrics[split].to_csv(REPORTS/f'notebook07_{split}_faiss_metrics.csv',index=False)
    rows=((u,rank,item) for u in results[split]['users']
          for rank,item in enumerate(results[split]['recs'][u],start=1))
    path=PROCESSED/f'notebook07_{split}_faiss_candidates.csv'
    pd.DataFrame(rows,columns=['visitorid','rank','itemid']).to_csv(path,index=False)
    print('Saved:',path)
summary={
    'method':'item2vec_mean_user_embedding_full_vocabulary_faiss_with_popularity_fallback',
    'protocol':'warm-user future-view pairs; exclude seen; cold/seen labels remain in denominator',
    'cutoffs_utc':{'validation':'2015-08-01','test':'2015-09-01'},
    'config':{'index_kind':INDEX_KIND,'hnsw_m':HNSW_M if INDEX_KIND=='hnsw' else None,
              'hnsw_ef_search':HNSW_EF_SEARCH if INDEX_KIND=='hnsw' else None,
              'seed_items_per_user':SEED_ITEMS_PER_USER,'search_k':SEARCH_K,
              'max_k':MAX_K,'query_batch_size':QUERY_BATCH_SIZE},
    'index_diagnostics':build_info,
}
for split,name in [('val','validation'),('test','test')]:
    r=results[split]
    summary[name]={'users':len(r['users']), 'positive_pairs':len(inputs[split]['labels']),
                   'eligible_pairs':r['eligible_pairs'],'batch_seconds':r['batch_seconds'],
                   'users_with_embedding_queries':r['users_with_embedding_queries'],
                   'users_with_embedding_candidates':r['users_with_embedding_candidates'],
                   'avg_embedding_candidates':r['avg_embedding_candidates'],
                   'metrics':metrics[split].to_dict(orient='records')}
output=REPORTS/'notebook07_faiss_retrieval.json'
output.write_text(json.dumps(summary,indent=2),encoding='utf-8')
assert all(metrics[s]['K'].tolist()==K_VALUES for s in ['val','test'])
assert all(metrics[s]['Recall@K_micro'].between(0,1).all() for s in ['val','test'])
assert all(results[s]['eligible_pairs']<=len(inputs[s]['labels']) for s in ['val','test'])
print('Saved:',output)
print('PASS: temporal boundaries, full-vocabulary FAISS, seen filtering, metric exports, and candidate exports')

Saved: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\data\processed\notebook07_val_faiss_candidates.csv
Saved: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\data\processed\notebook07_test_faiss_candidates.csv
Saved: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\reports\notebook07_faiss_retrieval.json
PASS: temporal boundaries, full-vocabulary FAISS, seen filtering, metric exports, and candidate exports


## 8. Interpret results and proceed to Notebook 08

1. **Compare Validation Recall@100** against Item-CF (6.24%) and restricted-seed Item2Vec (3.81%). Higher embedding-user coverage does not guarantee higher recall.
2. **Watch vocabulary coverage:** only items with at least five occurrences in Notebook 06’s Word2Vec training sequences have vectors. FAISS cannot fix missing embeddings.
3. **Observe catalog coverage:** the full index should allow more personalized items than the 3,000-seed restriction, but results are empirical.
4. **Do not call FAISS itself an accuracy improvement:** user-embedding averaging changes the scoring algorithm. Optional `INDEX_KIND='hnsw'` tests approximate retrieval; `flat` is exact.
5. **Next:** Notebook 08 — Hybrid Candidate Generation, combining Popularity, Item-CF, and FAISS with source flags and consistent validation metrics.
